
# Model Ops Lab Demo

Bayesian-flavoured model operations loop: a modelling agent iteratively updates a
posterior, an evaluation agent (LLM-backed) produces metric insights, a recommender
agent (LLM-backed) proposes improvements, and the modeller applies them until the
quality crosses 99. Explanations are logged each step and recommendations are
surfaced for review.


In [ ]:
# Cell 1: import libraries, seed RNG, and define knowledge bases/capabilities


import os
from copy import deepcopy
from random import Random

from agentic_codex import AgentBuilder, Context, FunctionAdapter
from agentic_codex.core.capabilities import (
    ContextCapability,
    MessageBusCapability,
    ResourceCapability,
    ToolCapability,
)
from agentic_codex.core.kernel import AgentKernel
from agentic_codex.core.memory import EpisodicMemory
from agentic_codex.core.schemas import AgentStep, Message
from agentic_codex.core.tools import MathToolAdapter, ToolPermissions, BudgetGuard
from agentic_codex.core.message_bus import MessageBus
from agentic_codex.core.llm import EnvOpenAIAdapter

rng = Random(123)

# Priors for candidate models (Beta alpha/beta) and descriptive features
MODEL_STORE = {
    "baseline": {"alpha": 45.0, "beta": 5.0, "features": ["bayes-beta", "dropout=0.2"]},
    "wide-net": {"alpha": 50.0, "beta": 6.0, "features": ["bayes-beta", "units=256"]},
    "distilled": {"alpha": 47.0, "beta": 7.0, "features": ["student", "temperature=2.0"]},
}

# Prompt context for the evaluation LLM
EVALUATION_KB = {
    "synthetic_boost": 2.5,
    "metrics": [
        {"name": "Posterior Mean", "goal": ">=0.99", "importance": "expected accuracy"},
        {"name": "Uncertainty", "goal": "≤0.01", "importance": "credible interval width"},
        {"name": "Calibration", "goal": ">=0.95", "importance": "probability quality"},
    ],
    "risk_tags": ["sensor drift", "heteroskedastic noise", "class imbalance"],
}

# Playbook entries and idioms used by the recommendation LLM
RECOMMENDATION_KB = {
    "playbooks": [
        {"threshold": 94, "action": "add calibration layer", "alpha": 4.0, "beta": -0.5},
        {"threshold": 96, "action": "expand augmentation suite", "alpha": 3.5, "beta": -0.2},
        {"threshold": 98, "action": "deploy ensemble voting", "alpha": 2.5, "beta": -0.1},
    ],
    "patterns": ["bayesian tuning", "feature crosses", "data augmentation pipelines"],
}

# Principles guiding stakeholder explanations
EXPLANATION_KB = {
    "principles": ["Reward empirical gains", "Surface data risks", "Tie to stakeholder goals"],
}

# Mapping from recommendation actions to posterior parameter adjustments
ACTION_EFFECTS = {
    "add calibration layer": {"alpha": 4.0, "beta": -0.5},
    "expand augmentation suite": {"alpha": 3.5, "beta": -0.2},
    "deploy ensemble voting": {"alpha": 2.5, "beta": -0.1},
    "baseline tuning": {"alpha": 2.0, "beta": -0.3},
}

# Tooling guards (permissions + budgets) for the modelling agent
permissions = ToolPermissions({"modeler": {"math"}})
budget_guard = BudgetGuard(max_calls=48)
math_tool = MathToolAdapter(name="math")

shared_bus = MessageBus()

context_capability = ContextCapability(data={"model_store": MODEL_STORE})
message_bus_factory = lambda: MessageBusCapability(bus=shared_bus)
tool_capability = ToolCapability(tools={"math": math_tool}, permissions=permissions, budget=budget_guard)
evaluator_knowledge = ResourceCapability(name="knowledge.evaluator", resource=EVALUATION_KB, target="components")
reco_knowledge = ResourceCapability(name="knowledge.recommendation", resource=RECOMMENDATION_KB, target="components")
explainer_knowledge = ResourceCapability(name="knowledge.explainer", resource=EXPLANATION_KB, target="components")
memory_capability = ResourceCapability(name="memory.loop", resource=EpisodicMemory(), target="components")

# Switch between real OpenAI adapter and stub fallback
OPENAI_MODEL = "gpt-4o-mini"
if os.getenv("OPENAI_API_KEY"):
    llm_adapter = EnvOpenAIAdapter(model=OPENAI_MODEL)
else:
    def _fallback(prompt: str) -> str:
        snippet = prompt.splitlines()[-1]
        return f"[stub llm] {snippet}"
    llm_adapter = FunctionAdapter(_fallback)


In [ ]:
# Cell 2: declare each agent (modeller, evaluator, recommender, improver, explainer)


# Utility: capture iteration/eval/recommendation/explanation logs
def ensure_logs(ctx: Context):
    return ctx.components.setdefault(
        "logs",
        {"iterations": [], "evaluations": [], "recommendations": [], "explanations": []},
    )


# Modeller agent lifecycle hooks
def modeler_init(ctx: Context) -> None:
    ctx.scratch.setdefault("iteration", 0)
    ctx.scratch.setdefault("history", [])
    ctx.scratch.setdefault("pending_actions", [])
    ensure_logs(ctx)


# Produce next posterior sample + quality estimate
def modeler_decide(ctx: Context) -> AgentStep:
    iteration = ctx.scratch["iteration"]
    model_store = ctx.scratch.setdefault("context", {}).setdefault("model_store", MODEL_STORE)
    chosen_name = list(model_store)[iteration % len(model_store)]
    state = ctx.scratch.setdefault("bayes_state", deepcopy(model_store[chosen_name]))
    actions = ctx.scratch.pop("pending_actions", [])
    for action in actions:
        effect = ACTION_EFFECTS.get(action, {"alpha": 2.0, "beta": -0.2})
        state["alpha"] = max(1.0, state["alpha"] + effect["alpha"] + rng.uniform(-0.2, 0.5))
        state["beta"] = max(1.0, state["beta"] + effect["beta"] + rng.uniform(-0.1, 0.2))
    alpha = state["alpha"]
    beta = state["beta"]
    posterior_mean = alpha / (alpha + beta)
    quality = max(0.0, min(100.0, posterior_mean * 100 + rng.uniform(-0.4, 0.6)))
    state["quality"] = quality
    payload = {
        "name": chosen_name,
        "state": deepcopy(state),
        "actions_applied": actions,
        "posterior_mean": posterior_mean,
    }
    ctx.scratch["current_model"] = payload
    ctx.scratch["model_summary"] = (
        f"Model {chosen_name} | alpha={alpha:.1f}, beta={beta:.1f}"
    )
    bus = ctx.get_message_bus()
    bus.publish(agent="modeler", content=f"{chosen_name} => quality {quality:.2f}", iteration=iteration, channel="model")
    message = Message(role="modeler", content=f"iteration {iteration} -> quality {quality:.2f}")
    return AgentStep(out_messages=[message], state_updates={"model": payload})


# Persist modeller output into the log
def modeler_learn(ctx: Context) -> None:
    logs = ensure_logs(ctx)
    record = {"iteration": ctx.scratch["iteration"], "model": ctx.scratch.get("current_model", {})}
    ctx.scratch.setdefault("history", []).append(record)
    logs["iterations"].append(record)


modeler_kernel = AgentKernel(decide_hook=modeler_decide, init_hook=modeler_init, learn_hook=modeler_learn)

modeler = (
    AgentBuilder(name="modeler", role="builder")
    .with_capabilities([message_bus_factory(), context_capability, tool_capability])
    .with_step(modeler_decide)
    .build(kernel=modeler_kernel)
)


# LLM-based evaluation of posterior metrics and risks
def evaluator_step(ctx: Context) -> AgentStep:
    knowledge = ctx.get_component("knowledge.evaluator")
    model = ctx.scratch["current_model"]
    state = model["state"]
    posterior_mean = state["quality"] / 100
    posterior_var = (posterior_mean * (1 - posterior_mean)) / (state["alpha"] + state["beta"] + 1)
    uncertainty = max(0.0, min(1.0, posterior_var ** 0.5))
    boost = knowledge.get("synthetic_boost", 2.5)
    adjusted_quality = max(0.0, min(100.0, state["quality"] - uncertainty * 100 + boost))
    ctx.scratch["quality"] = adjusted_quality
    metrics_brief = "
".join(
        f"- {m['name']} target {m['goal']} ({m['importance']})" for m in knowledge["metrics"]
    )
    prompt = (
        "You are an evaluation analyst critiquing a Bayesian model.
"
        f"Posterior mean: {posterior_mean:.3f}
"
        f"Uncertainty (std): {uncertainty:.3f}
"
        f"Adjusted quality: {adjusted_quality:.2f}
"
        f"Metrics: {metrics_brief}
"
        f"Risk tags: {', '.join(knowledge['risk_tags'])}
"
        "Provide concise insights and highlight risks."
    )
    insights = ctx.llm.generate(prompt)
    ctx.scratch["evaluation_insights"] = insights
    logs = ensure_logs(ctx)
    logs["evaluations"].append(
        {
            "iteration": ctx.scratch["iteration"],
            "quality": adjusted_quality,
            "uncertainty": uncertainty,
            "insights": insights,
        }
    )
    bus = ctx.get_message_bus()
    bus.publish(agent="evaluator", content=insights, iteration=ctx.scratch["iteration"], channel="analysis")
    return AgentStep(out_messages=[Message(role="evaluator", content=f"quality {adjusted_quality:.2f}")], state_updates={"quality": adjusted_quality, "evaluation_insights": insights})


evaluator = (
    AgentBuilder(name="evaluator", role="qa")
    .with_capabilities([message_bus_factory(), context_capability, evaluator_knowledge])
    .with_llm(llm_adapter)
    .with_step(evaluator_step)
    .build()
)


# LLM-based improvement suggestions using evaluation insights
def recommendation_step(ctx: Context) -> AgentStep:
    knowledge = ctx.get_component("knowledge.recommendation")
    quality = ctx.scratch["quality"]
    model_summary = ctx.scratch.get("model_summary", "")
    evaluation_insights = ctx.scratch.get("evaluation_insights", "")
    prompt = (
        "You are a model improvement recommender.
"
        f"Current quality: {quality:.2f}
"
        f"Model summary: {model_summary}
"
        f"Evaluation insights: {evaluation_insights}
"
        f"Playbooks: {knowledge['playbooks']}
"
        f"Patterns: {knowledge['patterns']}
"
        "Suggest up to 3 concrete actions with rationale."
    )
    recommendation_text = ctx.llm.generate(prompt)
    actions = []
    for entry in knowledge["playbooks"]:
        if quality < entry["threshold"]:
            actions.append(entry["action"])
    if not actions:
        actions.append("baseline tuning")
    ctx.scratch["pending_actions"] = actions
    logs = ensure_logs(ctx)
    logs["recommendations"].append(
        {
            "iteration": ctx.scratch["iteration"],
            "quality": quality,
            "actions": actions,
            "llm": recommendation_text,
        }
    )
    bus = ctx.get_message_bus()
    bus.publish(agent="recommender", content=recommendation_text, iteration=ctx.scratch["iteration"], channel="recommendations")
    return AgentStep(out_messages=[Message(role="recommender", content=", ".join(actions))], state_updates={"recommendation_text": recommendation_text, "actions": actions})


recommender = (
    AgentBuilder(name="recommender", role="advisor")
    .with_capabilities([message_bus_factory(), context_capability, reco_knowledge])
    .with_llm(llm_adapter)
    .with_step(recommendation_step)
    .build()
)


# Apply incremental quality nudges (simulating deployment)
def improvement_step(ctx: Context) -> AgentStep:
    model = ctx.scratch["current_model"]
    quality = ctx.scratch["quality"] + rng.uniform(0.3, 0.8)
    quality = min(100.0, quality)
    model["state"]["quality"] = quality
    ctx.scratch["quality"] = quality
    bus = ctx.get_message_bus()
    bus.publish(agent="improver", content=f"quality nudged to {quality:.2f}", iteration=ctx.scratch["iteration"], channel="operations")
    return AgentStep(out_messages=[Message(role="improver", content=f"quality now {quality:.2f}")], state_updates={"quality": quality})


improver = (
    AgentBuilder(name="improver", role="operations")
    .with_capabilities([message_bus_factory(), context_capability, memory_capability])
    .with_step(improvement_step)
    .build()
)


# LLM reviewer narrates the iteration for stakeholders
def explanation_step(ctx: Context) -> AgentStep:
    knowledge = ctx.get_component("knowledge.explainer")
    prompt = (
        "You are an LLM reviewer summarizing the iteration.
"
        f"Iteration: {ctx.scratch['iteration']}
"
        f"Quality: {ctx.scratch['quality']:.2f}
"
        f"Actions: {ctx.scratch.get('pending_actions', [])}
"
        f"Evaluation insights: {ctx.scratch.get('evaluation_insights', '')}
"
        f"Principles: {knowledge['principles']}
"
        "Create a brief explanation."
    )
    review = ctx.llm.generate(prompt)
    logs = ensure_logs(ctx)
    logs["explanations"].append({"iteration": ctx.scratch["iteration"], "explanation": review})
    bus = ctx.get_message_bus()
    bus.publish(agent="explainer", content=review, iteration=ctx.scratch["iteration"], channel="explanations")
    return AgentStep(out_messages=[Message(role="explainer", content=review)])


explainer = (
    AgentBuilder(name="explainer", role="reviewer")
    .with_capabilities([message_bus_factory(), context_capability, explainer_knowledge])
    .with_llm(llm_adapter)
    .with_step(explanation_step)
    .build()
)


In [ ]:
# Cell 3: orchestrate the iterative loop until quality ≥ 99


# Initialise context, attach message bus, and seed log structure
context = Context(goal="Elevate model quality to >= 99")
context.add_component("message_bus", shared_bus)
context.components.setdefault("logs", {"iterations": [], "evaluations": [], "recommendations": [], "explanations": []})
context.scratch["iteration"] = 0

# The five-agent assembly we iterate through
loop_agents = [modeler, evaluator, recommender, improver, explainer]
history = []
# Safety limit to avoid infinite loops in synthetic demo
MAX_ITERS = 18

def achieved_target() -> bool:
    return context.scratch.get("quality", 0.0) >= 99.0

# Run modeller → evaluator → recommender → improver → explainer loop
while not achieved_target() and context.scratch["iteration"] < MAX_ITERS:
    iteration = context.scratch["iteration"]
    for agent in loop_agents:
        result = agent.run(context)
        context.push_message(result.out_messages[-1])
# Track quality trajectory for post-run inspection
    history.append({"iteration": iteration, "quality": round(context.scratch.get("quality", 0.0), 3)})
    context.scratch["iteration"] += 1

final_quality = context.scratch.get("quality", 0.0)
final_model = context.scratch.get("current_model", {})
history, final_model, final_quality


In [ ]:
# Cell 4: collect structured logs for later inspection


# Summarise structured logs for quick inspection
logs = context.components["logs"]
logs_summary = {
    "iterations": logs["iterations"],
    "evaluations": logs["evaluations"],
    "recommendations": logs["recommendations"],
    "explanations": logs["explanations"],
}
logs_summary


In [ ]:
# Cell 5: view raw message bus chatter among agents


# Inspect raw message bus chatter (optional diagnostics)
bus_records = [
    {"agent": record.agent, "iter": record.iteration, "channel": record.channel, "message": record.content}
    for record in shared_bus.conversation()
]
bus_records[:20]
